# Auditoría simple de targets y conteos

Este notebook sirve para empezar desde números correctos antes de entrenar modelos. Usa como base la lógica de carga de `predictor_victim_20260608_FIXED.ipynb`, pero **no entrena ningún modelo**.

Objetivo:

1. Cargar `lista_global_vars.csv` y `target_col.csv`.
2. Contar los targets originales.
3. Aplicar el mismo filtrado usado en el notebook de víctimas: exclusión de `GENERO_BIN_2 == 1` y `ORIENTSEX.BN_3 == 1`.
4. Volver a contar targets después del filtrado.
5. Simular un split estratificado 75/25 para cada target y comprobar cuántos positivos/negativos quedan en train y test.
6. Guardar tablas CSV de auditoría.

La idea es responder preguntas como: `¿de dónde salen los 372 casos?`, `¿qué columna suma 372?`, `¿el target de victimización era VÍCTIMA amplia o POLIVICTIMIZACION?`.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)


## 1. Localizar y cargar los CSV

El notebook original usaba:

```python
feat_df = pd.read_csv('./../data/lista_global_vars.csv')
target_df = pd.read_csv('./../data/target_col.csv').fillna(0)
```

Aquí se prueban varias rutas habituales para que puedas ejecutarlo desde distintas carpetas.


In [ ]:
def find_first_existing(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError('No se encontró ninguno de estos ficheros:
' + '
'.join(map(str, candidates)))

feat_path = find_first_existing([
    '../data/lista_global_vars.csv',
    './data/lista_global_vars.csv',
    'data/lista_global_vars.csv',
    './lista_global_vars.csv',
])

target_path = find_first_existing([
    '../data/target_col.csv',
    './data/target_col.csv',
    'data/target_col.csv',
    './target_col.csv',
])

print('Features:', feat_path.resolve())
print('Targets :', target_path.resolve())

feat_df = pd.read_csv(feat_path)
target_df_raw = pd.read_csv(target_path)
target_df = target_df_raw.fillna(0)

print('
Dim características:', feat_df.shape)
print('Dim target raw    :', target_df_raw.shape)
print('Dim target fillna :', target_df.shape)
print('
Columnas target:')
print(target_df.columns.to_list())


## 2. Conteo directo de las columnas target

Primero contamos las columnas que te interesan **antes de aplicar ningún filtrado**.

Incluye tanto `VÍCTIMA` como posibles variantes (`VICTIMA`, sin tilde), por si existen en alguna versión de la base.


In [ ]:
target_cols_to_check = [
    'POLIVICTIMIZACION',
    'VÍCTIMA',
    'VICTIMA',
    'VICTIMA_PERPETRADOR',
    'PERPETRADOR',
    'POLIPERPETRACION',
    'SOLO.VICTIMA',
    'SOLO.PERPETRADOR',
    'NO.VICT_NO.PERP',
    'V.O',
    'P.SUM.TOTAL',
    'V.SUM.TOTAL',
]

existing_target_cols = [c for c in target_cols_to_check if c in target_df.columns]
missing_target_cols = [c for c in target_cols_to_check if c not in target_df.columns]

print('Columnas encontradas:', existing_target_cols)
print('Columnas NO encontradas:', missing_target_cols)

for col in existing_target_cols:
    print('
' + '='*80)
    print(f'target_df["{col}"].value_counts(dropna=False)')
    display(target_df[col].value_counts(dropna=False).sort_index())


## 3. Tabla resumen de targets antes del filtrado

Esta tabla busca automáticamente qué columnas binarias o numéricas suman valores cercanos a números clave como **372**, **221**, **178**, **621**, **950**, **771**, etc.


In [ ]:
def summarize_binary_like_columns(df, cols=None, name='dataset'):
    if cols is None:
        cols = df.columns
    rows = []
    for col in cols:
        s = df[col]
        if not pd.api.types.is_numeric_dtype(s):
            continue
        non_na = s.dropna()
        unique_vals = sorted(pd.unique(non_na))[:20]
        is_binary_like = set(pd.unique(non_na)).issubset({0, 1, 0.0, 1.0, False, True})
        rows.append({
            'dataset': name,
            'column': col,
            'n': len(s),
            'missing': int(s.isna().sum()),
            'unique_values_preview': str(unique_vals),
            'is_binary_like': is_binary_like,
            'sum': float(non_na.sum()) if len(non_na) else np.nan,
            'mean_pct': float(non_na.mean() * 100) if len(non_na) else np.nan,
            'count_0': int((non_na == 0).sum()),
            'count_1': int((non_na == 1).sum()),
        })
    return pd.DataFrame(rows)

summary_raw = summarize_binary_like_columns(target_df, name='target_df_raw_fillna')

# Mostrar primero columnas binarias y luego las más cercanas a los números que estamos buscando.
key_numbers = [372, 221, 178, 621, 950, 771, 1994, 184, 1223]
summary_raw['closest_key_number'] = summary_raw['sum'].apply(lambda x: min(key_numbers, key=lambda k: abs(k-x)) if pd.notna(x) else np.nan)
summary_raw['distance_to_key'] = summary_raw.apply(lambda r: abs(r['sum'] - r['closest_key_number']) if pd.notna(r['sum']) else np.nan, axis=1)

print('Columnas binarias o tipo target ordenadas por suma:')
display(summary_raw[summary_raw['is_binary_like']].sort_values('sum'))

print('
Columnas más cercanas a números clave:')
display(summary_raw.sort_values('distance_to_key').head(30))


## 4. Aplicar el mismo filtrado del notebook de víctimas

El notebook original hacía:

```python
df_merged = feat_df.join(target_df, how='inner')
df_merged = df_merged[~((df_merged["GENERO_BIN_2"] == 1) | (df_merged["ORIENTSEX.BN_3"] == 1))]
```

Este filtrado puede cambiar los conteos. Aquí lo replicamos y volvemos a contar.


In [ ]:
df_merged_initial = feat_df.join(target_df, how='inner')
print('Dim df_merged inicial:', df_merged_initial.shape)

required_filter_cols = ['GENERO_BIN_2', 'ORIENTSEX.BN_3']
missing_filter_cols = [c for c in required_filter_cols if c not in df_merged_initial.columns]

if missing_filter_cols:
    print('AVISO: no se puede aplicar el filtrado porque faltan columnas:', missing_filter_cols)
    df_merged_filtered = df_merged_initial.copy()
else:
    mask_excluded = (df_merged_initial['GENERO_BIN_2'] == 1) | (df_merged_initial['ORIENTSEX.BN_3'] == 1)
    print('Registros excluidos por GENERO_BIN_2 == 1 u ORIENTSEX.BN_3 == 1:', int(mask_excluded.sum()))
    df_merged_filtered = (
        df_merged_initial.loc[~mask_excluded]
        .drop(columns=['GENERO_BIN_2', 'ORIENTSEX.BN_3'])
        .reset_index(drop=True)
    )

print('Dim df_merged filtrado:', df_merged_filtered.shape)


In [ ]:
print('Conteos después del filtrado usado en el notebook:')

for col in existing_target_cols:
    if col in df_merged_filtered.columns:
        print('
' + '='*80)
        print(f'df_merged_filtered["{col}"].value_counts(dropna=False)')
        display(df_merged_filtered[col].value_counts(dropna=False).sort_index())

summary_filtered = summarize_binary_like_columns(df_merged_filtered, name='df_merged_filtered')
summary_filtered['closest_key_number'] = summary_filtered['sum'].apply(lambda x: min(key_numbers, key=lambda k: abs(k-x)) if pd.notna(x) else np.nan)
summary_filtered['distance_to_key'] = summary_filtered.apply(lambda r: abs(r['sum'] - r['closest_key_number']) if pd.notna(r['sum']) else np.nan, axis=1)

print('
Columnas binarias ordenadas por suma, después del filtrado:')
display(summary_filtered[summary_filtered['is_binary_like']].sort_values('sum'))

print('
Columnas más cercanas a números clave, después del filtrado:')
display(summary_filtered.sort_values('distance_to_key').head(30))


## 5. Buscar exactamente de dónde podrían salir 372, 221 y 178

Esta celda lista columnas cuya suma es exactamente o casi exactamente esos valores antes y después del filtrado.


In [ ]:
def find_columns_near_counts(summary_df, targets=(372, 221, 178), tolerance=10):
    out = []
    for target in targets:
        tmp = summary_df.copy()
        tmp['target_count'] = target
        tmp['distance'] = (tmp['sum'] - target).abs()
        tmp = tmp[tmp['distance'] <= tolerance].sort_values('distance')
        out.append(tmp)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

near_raw = find_columns_near_counts(summary_raw, targets=(372, 221, 178), tolerance=20)
near_filtered = find_columns_near_counts(summary_filtered, targets=(372, 221, 178), tolerance=20)

print('Columnas cercanas a 372, 221 o 178 ANTES del filtrado:')
display(near_raw[['dataset','column','sum','mean_pct','count_0','count_1','target_count','distance','unique_values_preview']])

print('
Columnas cercanas a 372, 221 o 178 DESPUÉS del filtrado:')
display(near_filtered[['dataset','column','sum','mean_pct','count_0','count_1','target_count','distance','unique_values_preview']])


## 6. Simular split 75/25 estratificado para cada target

Esto permite comprobar si un test de 942 casos puede tener tantos positivos como el que estamos viendo.

Para cada columna binaria disponible, se hace:

```python
train_test_split(index, test_size=0.25, random_state=42, stratify=target)
```

Y se guardan los soportes de train/test.


In [ ]:
def split_supports_for_targets(df, target_cols, test_size=0.25, random_state=42):
    rows = []
    for col in target_cols:
        if col not in df.columns:
            continue
        y = df[col].fillna(0)
        vals = set(pd.unique(y))
        if not vals.issubset({0, 1, 0.0, 1.0, False, True}):
            continue
        try:
            idx_train, idx_test = train_test_split(
                df.index,
                test_size=test_size,
                random_state=random_state,
                stratify=y
            )
            y_train = y.loc[idx_train]
            y_test = y.loc[idx_test]
            rows.append({
                'column': col,
                'n_total': len(y),
                'total_0': int((y == 0).sum()),
                'total_1': int((y == 1).sum()),
                'prev_total_pct': float((y == 1).mean() * 100),
                'n_train': len(y_train),
                'train_0': int((y_train == 0).sum()),
                'train_1': int((y_train == 1).sum()),
                'prev_train_pct': float((y_train == 1).mean() * 100),
                'n_test': len(y_test),
                'test_0': int((y_test == 0).sum()),
                'test_1': int((y_test == 1).sum()),
                'prev_test_pct': float((y_test == 1).mean() * 100),
            })
        except Exception as e:
            rows.append({
                'column': col,
                'error': str(e)
            })
    return pd.DataFrame(rows)

split_summary = split_supports_for_targets(df_merged_filtered, existing_target_cols)
display(split_summary)


## 7. Comprobar la definición de los cuatro grupos de rol

Si existen estas columnas:

- `NO.VICT_NO.PERP`
- `SOLO.VICTIMA`
- `SOLO.PERPETRADOR`
- `VICTIMA_PERPETRADOR`

esta celda comprueba si son excluyentes y si suman 1 por fila.


In [ ]:
role_cols = ['NO.VICT_NO.PERP', 'SOLO.VICTIMA', 'SOLO.PERPETRADOR', 'VICTIMA_PERPETRADOR']
role_cols_existing = [c for c in role_cols if c in df_merged_filtered.columns]

print('Columnas de rol encontradas:', role_cols_existing)

if len(role_cols_existing) == len(role_cols):
    role_sum = df_merged_filtered[role_cols].sum(axis=1)
    print('Distribución de suma por fila en columnas de rol:')
    display(role_sum.value_counts(dropna=False).sort_index())
    print('
Conteos por grupo:')
    display(df_merged_filtered[role_cols].sum().to_frame('n').assign(pct=lambda d: d['n'] / len(df_merged_filtered) * 100))
else:
    print('No están todas las columnas de rol; se omite esta comprobación.')


## 8. Guardar resultados de auditoría

Se guardan varios CSV en `./audit_targets_output/`.


In [ ]:
output_dir = Path('./audit_targets_output')
output_dir.mkdir(parents=True, exist_ok=True)

summary_raw.to_csv(output_dir / 'summary_targets_raw_fillna.csv', index=False)
summary_filtered.to_csv(output_dir / 'summary_targets_after_filter.csv', index=False)
split_summary.to_csv(output_dir / 'split_supports_after_filter.csv', index=False)
near_raw.to_csv(output_dir / 'columns_near_372_221_178_raw.csv', index=False)
near_filtered.to_csv(output_dir / 'columns_near_372_221_178_after_filter.csv', index=False)

print('Archivos guardados en:', output_dir.resolve())
for p in sorted(output_dir.glob('*.csv')):
    print('-', p.name)


## 9. Lectura rápida esperada

Cuando ejecutes el notebook, mira especialmente:

1. `target_df["VÍCTIMA"].value_counts()` para saber si el notebook estaba usando victimización amplia.
2. `target_df["POLIVICTIMIZACION"].value_counts()` para saber si el criterio de polivictimización se acerca a 372.
3. La tabla **Columnas cercanas a 372, 221 o 178**.
4. La tabla **split_summary**, para comprobar cuántos positivos quedarían en el test para cada target.

Si `VÍCTIMA` tiene aproximadamente 1.860 positivos tras el filtrado, entonces el modelo de víctima que teníamos estaba entrenando sobre victimización amplia, no sobre polivictimización.
